# Spatial Algebra
The form of [](#eq-newtoneuler) implies dynamics of rigid bodies are a concatenation of equations on the linear/translational and the rotational parts. It hence makes lots of sense to consider linear and rotational parts as a combined quantity. Actually, if we define:

$$
\mathcal{F} = \begin{bmatrix}
\mathbf{\tau} \\
\mathbf{f}
\end{bmatrix}
\quad
\mathcal{V} = \begin{bmatrix}
\mathbf{\omega} \\
\mathbf{v}
\end{bmatrix}
\quad
\mathcal{M} = \begin{bmatrix}
\mathcal{I}  &   \mathbf{0}  \\
\mathbf{0}  &   m\mathbf{I}_{3\times 3}
\end{bmatrix}
\quad
\mathcal{V} \times = -(\mathcal{V} \times^*)^\top = \begin{bmatrix}
\mathbf{\omega}\times  &   \mathbf{0}  \\
\mathbf{v}\times &   \mathbf{\omega}\times
\end{bmatrix}
$$

We can get a compact form of [](#eq-newtoneuler) as

$$
\mathcal{F} = \mathcal{M}\dot{\mathcal{V}} + \mathcal{V} \times^* \mathcal{M}\mathcal{V}
$$

Note that this has a strong resemblence of the sole rotational part $\mathbf{\tau} = \mathcal{I}\dot{\mathbf{\omega}} + \mathbf{\omega}\times \mathcal{I}\mathbf{\omega}$. Rotational motion is more fundamental comparing to linear/translational component in rigid body motion, since the latter can be regarded as rotating about an axis infinitely far away. 

The above arrays like $\mathcal{V}$ and $\mathcal{F}$ are called __spatial vector__ for their main usage in describing spatial motion of rigid bodies. They are clearly generalized version for motion and force related to rigid bodies and also named __twist__ and __wrench__ respectively. The spatial vectors and the operations to transform them, e.g. $\mathcal{M}$ and $(\mathcal{V} \times)$, comprise __spatial algebra__, in a way similar as linear algebra consisted of vectors and operations like matrix-vector multiplication and dot/cross products. 

Spatial Algebra provides a concise way of describing rigid body dynamics. It also simplifies the computation much with the stationary reference frames. For what the stationary reference frames mean, see chapter 2.2 in [@featherstone2007] and page 58-60 in [@modernrobotics]. The reference to stationary frames allows for simple summation of spatial velocities, accelerations or inertias, which would be very convenient for adding relative quantities and compositing connected rigid bodies. The operations also generalize the counterparts with 3D vectors by retaining the same physics meaning, such as converting motion vectors to force vectors via inertia. This makes it intuitive to grasp the overall idea and algorithm steps. 

Similar to linear algebra, using Spatial Algebra to get correct results must ensure same frames for referencing and expressing spatial vectors involved in the operation. For instance, the spatial inertia is often given about the center-of-mass while the motion deduced from joint displacement is often with the joint frame. Coordinate transform is needed before the actual multiplication takes place. The transforms are again generalization to the 3D homogenous transformation matrix. For instance, following the convention of [@modernrobotics], the coordinate transform for motion vectors with reference to two frames associated by $\mathbf{T} = (\mathbf{R}, \mathbf{p})$ is:

$$
\mathcal{T} = \begin{bmatrix}
\mathbf{R}  &   \mathbf{0}  \\
\mathbf{p} \times \mathbf{R} &  \mathbf{R}
\end{bmatrix}
$$

And similar to the motion-force cross product $\mathcal{v} \times^* = -(\mathcal{v} \times)^\top$. The transform for spatial force vector is $\mathcal{T}^* = (\mathcal{T}^\top)^{-1}$. See chapter 2.8 in [@featherstone2007] for the list of transforms.

:::{note}
The conventions adopted to parameterize transformation might be different. See and compare chapter 3 in [@modernrobotics] and chapter 2 in [@featherstone2007]. In what follows, we choose to focus on the physical quantities without explicitly including coordination transform steps for the brevity of presentation.
:::





# Recursive Computation for Serial Link Velocities

With the spatial algebra notations, one can easily write down the relation between link velocities induced by the joint velocities. This is similar to forward kinematics assocating link poses to joint position $\mathbf{q}$, and can also be computed via a recursive process. As illustrated in [](#linkvel), the velocity of link $L_i$ is the combined effects of the predecessor link $L_{i-1}$ and the revoluting rate $\dot{q}_i$ along the screwing axis $\mathcal{S}_i$, which writes:

$$
\mathcal{V}_i = \mathcal{V}_{i, i-1} + \dot{q}_i\mathcal{S}_i
$$

Remember that $\mathcal{V}_{i, i-1}$ actually transforms from the previous $\mathcal{V}_{i-1}$ via ${}^{i-1}\mathbf{T}_{i}$ so the computation involves $q_i$. The screw axis $\mathcal{S}_i$ is simply a unit spatial velocity vector, e.g. for the x-axis referenced in the $L_i$ frame $\mathcal{S}_i = [1, 0, 0, 0, 0, 0]^\top$. By knowing ${}^{i-1}\mathbf{T}_{i}$, the basis link velocity ($\mathcal{V}_0 = \mathbf{0}$ if the robot basis is fixed) and recursively applying the above equation from the basis to the outermost link, one can obtain $\mathcal{V}_{i}$ for all the links, as well as $\mathcal{V}_{0, i}$ with ${}^0\mathbf{T}_{i}$ if the quantity in the operational space is interested. Unsurprisingly, $\mathcal{V}_i$ (and $\mathcal{V}_{0, i}$) are linear with $\dot{\mathbf{q}}$ so the above process can construct the geometry Jacobian in [kinematics](../2_LabVisionData/notebooks.ipynb).

```{figure} ../images/linkvel.png
---
name: linkvel
---

Relating the velocity of a successor link to its predecessor link and the relative velocity due to the joint revolution.
```

One may also come up with a similar recursive relation to compute link accelerations, when $\mathbf{q}$, $\dot{\mathbf{q}}$ and $\ddot{\mathbf{q}}$ are given. By differentiating both sides of [](#eq-linkvel), one can obtain:

$$
\dot{\mathcal{V}}_i = \dot{\mathcal{V}}_{i, i-1} + \ddot{q}_i\mathcal{S}_i + \dot{q}_i \mathcal{V}_i \times \mathcal{S}_i 
$$

Note the last term is due to $\dot{\mathcal{S}}_i = \mathcal{V}_i \times \mathcal{S}_i$, similar to differentiating vector components in $\mathbf{R}$. $\dot{\mathcal{V}}_{i, i-1}$ is simply transforming $\dot{\mathcal{V}}_{i-1}$ via ${}^{i-1}\mathbf{T}_{i}$ (Remember the frames are stationary at the instantaneous moment). To account for acceleration due to the gravity, one can start with $\dot{\mathcal{V}}_{0} = [0, 0, 0, 0, 0, -g]^\top$ if $-z$ is deemed as the gravity direction. The recursion definitely builds upon the recursion for $\mathcal{V}_i$, which in turn relies on the recursion in forward kinematics. Nonetheless, the complexity is still $O(N)$ with yet another scan from the basis to the outermost $N$-th link. These processes update link velocity and acceleration, and are integral parts of forward and inverse dynamics algorithms. 
